# Day 5 — SQL Practice Questions: All Joins

| # | Difficulty | Join Type | Topic |
|---|-----------|-----------|-------|
| 1 | 🟢 Easy   | INNER     | Orders with customer names |
| 2 | 🟢 Easy   | LEFT + IS NULL | Customers with no orders |
| 3 | 🟡 Medium | LEFT + GROUP BY | Total orders + revenue per customer |
| 4 | 🟢 Easy   | SELF      | Employee → manager hierarchy |
| 5 | 🟡 Medium | FULL OUTER | Unmatched rows on both sides |
| 6 | 🟢 Easy   | CROSS     | All customer–product combinations |

> **Run order:** top to bottom — setup cell creates all tables automatically

In [ ]:
%load_ext sql
USERNAME = 'postgres'
PASSWORD = 'hariom'
HOST     = 'localhost'
PORT     = '5432'
DBNAME   = 'postgres'
%sql postgresql://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}
print('Connected')

## Setup — Create and Populate Tables

In [ ]:
%%sql
DROP TABLE IF EXISTS d5_orders    CASCADE;
DROP TABLE IF EXISTS d5_customers CASCADE;
DROP TABLE IF EXISTS d5_employees CASCADE;
DROP TABLE IF EXISTS d5_products  CASCADE;

CREATE TABLE d5_customers (
    customer_id  SERIAL PRIMARY KEY,
    name         VARCHAR(50),
    email        VARCHAR(100),
    city         VARCHAR(50)
);
INSERT INTO d5_customers (name, email, city) VALUES
    ('Alice', 'alice@example.com', 'New York'),
    ('Bob',   'bob@example.com',   'Chicago'),
    ('Carol', 'carol@example.com', 'Houston'),
    ('Dave',  'dave@example.com',  'Phoenix'),
    ('Eve',   'eve@example.com',   'Seattle');

-- customer_id=99 is orphan; customer_id=4 (Dave) has no orders
CREATE TABLE d5_orders (
    order_id     SERIAL PRIMARY KEY,
    customer_id  INT,
    amount       NUMERIC(10,2),
    status       VARCHAR(20),
    order_date   DATE
);
INSERT INTO d5_orders (customer_id, amount, status, order_date) VALUES
    (1,  250.00, 'completed', '2024-01-05'),
    (1,  125.50, 'pending',   '2024-01-12'),
    (2,   89.99, 'completed', '2024-01-07'),
    (3,  450.00, 'completed', '2024-01-10'),
    (3,  210.00, 'cancelled', '2024-01-14'),
    (5,  320.00, 'completed', '2024-01-08'),
    (99,  75.00, 'pending',   '2024-01-15');

CREATE TABLE d5_employees (
    emp_id      SERIAL PRIMARY KEY,
    name        VARCHAR(50),
    role        VARCHAR(50),
    manager_id  INT
);
INSERT INTO d5_employees (name, role, manager_id) VALUES
    ('Sarah', 'CEO',            NULL),
    ('Tom',   'VP Engineering',    1),
    ('Priya', 'VP Marketing',      1),
    ('Alice', 'Engineer',          2),
    ('Bob',   'Engineer',          2),
    ('Carol', 'Designer',          3),
    ('Dave',  'Analyst',           3);

CREATE TABLE d5_products (
    product_id    SERIAL PRIMARY KEY,
    product_name  VARCHAR(50),
    category      VARCHAR(30),
    price         NUMERIC(8,2)
);
INSERT INTO d5_products (product_name, category, price) VALUES
    ('Widget A',  'Hardware',  29.99),
    ('Widget B',  'Hardware',  49.99),
    ('Service X', 'Software',  99.00),
    ('Service Y', 'Software', 149.00);

SELECT 'd5_customers' AS tbl, COUNT(*) AS rows FROM d5_customers
UNION ALL SELECT 'd5_orders',    COUNT(*) FROM d5_orders
UNION ALL SELECT 'd5_employees', COUNT(*) FROM d5_employees
UNION ALL SELECT 'd5_products',  COUNT(*) FROM d5_products;

---
# Question 1 🟢 — INNER JOIN: Orders with Customer Names

## Concept Warm-up

In [ ]:
%%sql
SELECT * FROM d5_customers ORDER BY customer_id;

In [ ]:
%%sql
SELECT * FROM d5_orders ORDER BY order_id;

In [ ]:
%%sql
-- Warm-up: basic JOIN syntax with table aliases
SELECT o.order_id, o.customer_id, c.name
FROM   d5_orders o
JOIN   d5_customers c ON o.customer_id = c.customer_id
LIMIT  3;

## Problem

Write a query that returns **all orders that have a matching customer**.  
Include: `order_id`, `customer name`, `city`, `amount`, `status`.  
Sort by `order_id` ascending.

**Expected:** 6 rows (the orphan order with `customer_id=99` should be excluded)

In [ ]:
%%sql
-- YOUR QUERY HERE


---
# Question 2 🟢 — LEFT JOIN: Customers with No Orders

## Concept Warm-up

In [ ]:
%%sql
-- Warm-up 1: LEFT JOIN — all customers + matching orders
SELECT c.name, o.order_id
FROM   d5_customers c
LEFT   JOIN d5_orders o ON c.customer_id = o.customer_id
ORDER  BY c.customer_id, o.order_id;

In [ ]:
%%sql
-- Warm-up 2: IS NULL detects unmatched rows — notice Dave
SELECT c.name, o.order_id,
       CASE WHEN o.order_id IS NULL THEN 'NO ORDERS' ELSE 'has orders' END AS check
FROM   d5_customers c
LEFT   JOIN d5_orders o ON c.customer_id = o.customer_id
ORDER  BY c.name;

In [ ]:
%%sql
-- Warm-up 3: WHERE IS NULL keeps only non-matched rows
SELECT c.name
FROM   d5_customers c
LEFT   JOIN d5_orders o ON c.customer_id = o.customer_id
WHERE  o.order_id IS NULL;

## Problem

Find all customers who have **placed zero orders**.  
Include: `customer_id`, `name`, `email`.  

**Expected:** 1 row — Dave (customer_id=4)

In [ ]:
%%sql
-- YOUR QUERY HERE


---
# Question 3 🟡 — LEFT JOIN + GROUP BY: Revenue per Customer

## Concept Warm-up

In [ ]:
%%sql
-- Warm-up 1: COUNT on nullable column — 0 for customers with no orders
SELECT c.name, COUNT(o.order_id) AS cnt
FROM   d5_customers c
LEFT   JOIN d5_orders o ON c.customer_id = o.customer_id
GROUP  BY c.customer_id, c.name
ORDER  BY c.name;

In [ ]:
%%sql
-- Warm-up 2: SUM is NULL when no rows — COALESCE replaces with 0
SELECT c.name,
       SUM(o.amount)              AS raw_sum,
       COALESCE(SUM(o.amount), 0) AS safe_sum
FROM   d5_customers c
LEFT   JOIN d5_orders o ON c.customer_id = o.customer_id
GROUP  BY c.customer_id, c.name
ORDER  BY c.name;

In [ ]:
%%sql
-- Warm-up 3: ORDER BY aggregate DESC
SELECT c.name, COALESCE(SUM(o.amount), 0) AS total
FROM   d5_customers c
LEFT   JOIN d5_orders o ON c.customer_id = o.customer_id
GROUP  BY c.customer_id, c.name
ORDER  BY total DESC;

## Problem

For **every customer** (including those with no orders), compute:
- `total_orders` — count of orders placed
- `total_revenue` — sum of amounts (show `0` not NULL if no orders)

Columns: `customer_id`, `name`, `total_orders`, `total_revenue`  
Sort by `total_revenue` **descending**.

**Expected:** 5 rows — Dave should appear with `total_orders=0`, `total_revenue=0`

In [ ]:
%%sql
-- YOUR QUERY HERE


---
# Question 4 🟢 — SELF JOIN: Employee → Manager

## Concept Warm-up

In [ ]:
%%sql
SELECT * FROM d5_employees ORDER BY emp_id;

In [ ]:
%%sql
-- Warm-up 2: alias the same table twice
SELECT e.name AS employee, e.manager_id
FROM   d5_employees e
ORDER  BY e.emp_id;

In [ ]:
%%sql
-- Warm-up 3: INNER SELF JOIN — only employees who have a manager
SELECT e.name AS employee, m.name AS manager
FROM   d5_employees e
INNER  JOIN d5_employees m ON e.manager_id = m.emp_id
ORDER  BY e.name;

## Problem

List **all employees** with their manager's name.  
The CEO (no manager) should still appear — show `NULL` for manager.  
Columns: `employee`, `role`, `manager` (NULL for CEO).  
Sort by `emp_id`.

In [ ]:
%%sql
-- YOUR QUERY HERE


---
# Question 5 🟡 — FULL OUTER JOIN: Detect Unmatched Rows

## Concept Warm-up

In [ ]:
%%sql
-- Warm-up 1: FULL OUTER JOIN returns everything from both sides
SELECT c.customer_id, c.name, o.order_id, o.customer_id AS o_cid
FROM   d5_customers c
FULL   OUTER JOIN d5_orders o ON c.customer_id = o.customer_id
ORDER  BY c.customer_id NULLS LAST, o.order_id;

In [ ]:
%%sql
-- Warm-up 2: label which side is unmatched
SELECT CASE
           WHEN c.customer_id IS NULL THEN 'Orphan order'
           WHEN o.order_id    IS NULL THEN 'Customer no orders'
           ELSE 'Matched'
       END AS match_status,
       COUNT(*)
FROM   d5_customers c
FULL   OUTER JOIN d5_orders o ON c.customer_id = o.customer_id
GROUP  BY 1;

In [ ]:
%%sql
-- Warm-up 3: WHERE with OR to keep only unmatched rows
SELECT c.name, o.order_id
FROM   d5_customers c
FULL   OUTER JOIN d5_orders o ON c.customer_id = o.customer_id
WHERE  c.customer_id IS NULL OR o.order_id IS NULL;

## Problem

Find all **unmatched rows** — rows that don't have a corresponding entry on the other side.  
Include both unmatched customers (no orders) AND orphan orders (no customer).  
Add a `reason` column: `'Customer with no orders'` or `'Orphan order'`.  
Columns: `customer_name`, `order_id`, `amount`, `reason`.

**Expected:** 2 rows — Dave (no orders) + the order with customer_id=99

In [ ]:
%%sql
-- YOUR QUERY HERE


---
# Question 6 🟢 — CROSS JOIN: All Customer–Product Combinations

## Concept Warm-up

In [ ]:
%%sql
SELECT * FROM d5_products ORDER BY product_id;

In [ ]:
%%sql
-- Warm-up 2: rows_left × rows_right
SELECT COUNT(*) AS total_combinations
FROM   d5_customers
CROSS  JOIN d5_products;

In [ ]:
%%sql
-- Warm-up 3: CROSS JOIN with WHERE to filter combinations
SELECT c.name, p.product_name, p.category
FROM   d5_customers c
CROSS  JOIN d5_products p
WHERE  p.category = 'Software'
ORDER  BY c.name, p.product_name;

## Problem

Generate a **catalog** of all possible customer–product pairings.  
Include: `customer_name`, `product_name`, `category`, `price`.  
Sort by `customer_name` ASC, then `product_name` ASC.

**Expected:** 20 rows (5 customers × 4 products)

In [ ]:
%%sql
-- YOUR QUERY HERE
